# This script scrapes match data from understatapi and saves it to a CSV file.

In [1]:
import pandas as pd
from understatapi import UnderstatClient
import time
from tqdm import tqdm
import json

### Inspect raw data from UnderStat

In [ ]:
def inspect_raw_data(league: str, season: str, sample_size: int = 1):
    """
    Inspect the raw data structure from understatapi to understand columns and format.
    
    Args:
        league (str): The league code (e.g., 'EPL')
        season (str): The season year (e.g., '2025')
        sample_size (int): Number of matches to inspect
    """
    with UnderstatClient() as understat:
        fixtures = understat.league(league=league).get_match_data(season=season)
        match_ids = [int(match['id']) for match in fixtures if match.get('isResult') is True]
        print(len(match_ids))
        
        print(f"Inspecting {sample_size} match(es) from {league} {season}...")
        
        for idx, match_id in enumerate(match_ids[:sample_size]):
            try:
                match_endpoint = understat.match(match=str(match_id))
                raw_shots = match_endpoint.get_shot_data(timeout=30)
                raw_context = match_endpoint.get_match_info(timeout=30)
                
                print(f"\n{'='*10}")
                print(f"Match ID: {match_id}")
                print(f"{'='*10}")
                
                print("\n--- RAW SHOTS DATA ---")
                print(f"Type: {type(raw_shots)}")
                if isinstance(raw_shots, list) and len(raw_shots) > 0:
                    print(f"Number of shots: {len(raw_shots)}")
                    print(f"First shot keys: {raw_shots[0].keys()}")
                    print(f"Sample shot:\n{json.dumps(raw_shots[0], indent=2)}")
                
                print("\n--- RAW CONTEXT DATA ---")
                print(f"Type: {type(raw_context)}")
                if isinstance(raw_context, dict):
                    print(f"Keys: {raw_context.keys()}")
                    print(f"Sample context:\n{json.dumps(raw_context, indent=2, default=str)}")
                    
            except Exception as exc:
                print(f"\n[Warning] Could not inspect Match ID {match_id}: {exc}")
            
            if idx < sample_size - 1:
                time.sleep(0.4)

inspect_raw_data(league='EPL', season='2025', sample_size=2)

368
Inspecting 2 match(es) from EPL 2025...

Match ID: 28778

--- RAW SHOTS DATA ---
Type: <class 'dict'>

--- RAW CONTEXT DATA ---
Type: <class 'dict'>
Keys: dict_keys(['id', 'fid', 'h', 'a', 'date', 'league_id', 'season', 'h_goals', 'a_goals', 'team_h', 'team_a', 'h_xg', 'a_xg', 'h_w', 'h_d', 'h_l', 'league', 'h_shot', 'a_shot', 'h_shotOnTarget', 'a_shotOnTarget', 'h_deep', 'a_deep', 'a_ppda', 'h_ppda', 'isData'])
Sample context:
{
  "id": "28778",
  "fid": "1903117",
  "h": "87",
  "a": "73",
  "date": "2025-08-15 19:00:00",
  "league_id": "1",
  "season": "2025",
  "h_goals": "4",
  "a_goals": "2",
  "team_h": "Liverpool",
  "team_a": "Bournemouth",
  "h_xg": "2.33007",
  "a_xg": "1.57303",
  "h_w": "0.5498",
  "h_d": "0.2276",
  "h_l": "0.2226",
  "league": "EPL",
  "h_shot": "19",
  "a_shot": "10",
  "h_shotOnTarget": "10",
  "a_shotOnTarget": "3",
  "h_deep": "9",
  "a_deep": "8",
  "a_ppda": "11.5833",
  "h_ppda": "8.7647",
  "isData": true
}

Match ID: 28779

--- RAW SHOTS DAT

### Scrape match data and save the raw data to a `csv` file.

In [ ]:
def scrape_match_data(league: str, season: str):
    """
    Scrapes shot-level data from matches and saves to CSV.
    
    Args:
        league (str): The league code (e.g., 'EPL')
        season (str): The season year (e.g., '2025')
    """
    with UnderstatClient() as understat:
        fixtures = understat.league(league=league).get_match_data(season=season)
        match_ids = [int(match['id']) for match in fixtures if match.get('isResult') is True]
        print(f'Found {len(match_ids)} completed {league} matches for season {season}.')

        records = []
        for match_id in tqdm(match_ids, desc='Parsing Matches'):
            try:
                match_endpoint = understat.match(match=str(match_id))
                raw_shots = match_endpoint.get_shot_data(timeout=30)
                raw_context = match_endpoint.get_match_info(timeout=30)
                
                # Extract match-level context
                if isinstance(raw_shots, dict):
                    home_team = raw_context.get("team_h")
                    away_team = raw_context.get("team_a")

                    for side, shots in raw_shots.items():  # side is usually "h" or "a"
                        team = home_team if side == "h" else away_team

                        for shot in shots:
                            row = shot.copy()
                            row["match_id"] = match_id
                            row["date"] = raw_context.get("date")
                            row["home_team"] = home_team
                            row["away_team"] = away_team
                            row["team"] = team
                            row["is_home"] = side == "h"
                            records.append(row)
                        
            except Exception as exc:
                print(f"\n[Warning] Could not extract data for Match ID {match_id}: {exc}")
            
            time.sleep(0.4)

        # Create DataFrame and save
        if records:
            dataset = pd.DataFrame(records)
            output_file = f'{league}_{season}_shots.csv'
            dataset.to_csv(output_file, index=False)
            print(f'Saved {dataset.shape[0]} rows and {dataset.shape[1]} columns to {output_file}.')
        else:
            print("No records scraped.")

scrape_match_data(league='EPL', season='2025')

Found 368 completed EPL matches for season 2025.


Parsing Matches: 100%|██████████| 368/368 [04:36<00:00,  1.33it/s]


Saved 9190 rows and 24 columns to EPL_2025_shots.csv.
